# Notebook 01 (Real) - Thu thap du lieu nha dat Ha Noi

Muc tieu:
- Thu thap du lieu that ve nha dat khu vuc Ha Noi.
- Tang quy mo thu thap (de sau cleaning con du mau train).
- Luu du lieu tho vao `data/raw/housing_raw_vn_hanoi_deep.csv`.

## 1) Cai dat thu vien (neu chua co)

Neu bi bao thieu thu vien, bo comment dong lenh ben duoi va chay lai.

In [6]:
%pip install requests beautifulsoup4 lxml pandas

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.7 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.7 kB ? eta -:--:--
     -------------------------------------- 41.7/41.7 kB 286.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
   ------------------------- -------------- 41.0/64.9 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 64.9/64.9 kB 1.2 MB/s eta 0:00:00
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   - -------------------------------------- 0.1/4.0 MB 6.8 MB/s eta 0:00:01
   - -------------------------------------- 0.2/4.0 MB 2.1 MB/s eta 0:00:02
   -- ------------------------------------- 0.3/4.0 MB 2.0 MB/s eta 0:00:02
   --- ------------------------------------ 0.


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, RAW_DIR

(WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha'),
 WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/raw'))

## 2) Cau hinh crawl

In [2]:
TARGET_ROWS = 4000
PAGE_SIZE = 50
MAX_PAGES = 120

# API public listing cua Cho Tot (nha dat)
API_URL = "https://gateway.chotot.com/v1/public/ad-listing"

# Ma khu vuc Ha Noi thuong dung tren endpoint listing
HANOI_REGION_V2 = 12000

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.chotot.com/",
    "Origin": "https://www.chotot.com"
}

TARGET_ROWS, PAGE_SIZE, MAX_PAGES

(4000, 50, 120)

## 3) Ham ho tro boc tach du lieu

In [3]:
def pick_text(node, selectors):
    for css in selectors:
        found = node.select_one(css)
        if found:
            txt = found.get_text(" ", strip=True)
            if txt:
                return txt
    return None


def pick_href(node, selectors):
    for css in selectors:
        found = node.select_one(css)
        if found and found.has_attr("href"):
            href = found["href"].strip()
            if href:
                if href.startswith("/"):
                    return f"https://batdongsan.com.vn{href}"
                return href
    return None

## 4) Crawl danh sach tin

In [4]:
rows = []
seen_ids = set()
session = requests.Session()

for page in range(1, MAX_PAGES + 1):
    if len(rows) >= TARGET_ROWS:
        break

    offset = (page - 1) * PAGE_SIZE
    params = {
        "region_v2": HANOI_REGION_V2,
        "cg": 1020,  # nha dat
        "limit": PAGE_SIZE,
        "o": offset,
        "st": "s,k",
        "sp": 0,
    }

    try:
        resp = session.get(API_URL, params=params, headers=HEADERS, timeout=25)
        if resp.status_code != 200:
            print(f"[Page {page}] status {resp.status_code} -> bo qua")
            continue

        payload = resp.json()
        ads = payload.get("ads", [])
    except Exception as e:
        print(f"[Page {page}] loi request/json: {e}")
        continue

    print(f"[Page {page}] tim thay {len(ads)} tin")

    if not ads:
        continue

    for ad in ads:
        ad_id = ad.get("list_id") or ad.get("ad_id")
        if ad_id in seen_ids:
            continue
        seen_ids.add(ad_id)

        district = ad.get("district_name")
        ward = ad.get("ward_name")
        location = ", ".join([x for x in [ward, district, "Ha Noi"] if x])

        rows.append({
            "source": "chotot",
            "city": "Ha Noi",
            "title": ad.get("subject"),
            "price_raw": ad.get("price_string") or ad.get("price"),
            "area_raw": ad.get("size") or ad.get("area"),
            "location_raw": location if location else None,
            "detail_url": ad.get("webp_image") or ad.get("image") or None,
            "list_id": ad_id,
            "category_name": ad.get("category_name"),
            "crawl_page": page,
            "crawl_ts": pd.Timestamp.now(tz="Asia/Ho_Chi_Minh")
        })

        if len(rows) >= TARGET_ROWS:
            break

    time.sleep(random.uniform(0.8, 1.6))

print(f"\nTong so ban ghi thu duoc: {len(rows)}")

[Page 1] tim thay 50 tin
[Page 2] tim thay 50 tin
[Page 3] tim thay 50 tin
[Page 4] tim thay 50 tin
[Page 5] tim thay 50 tin
[Page 6] tim thay 50 tin
[Page 7] tim thay 50 tin
[Page 8] tim thay 50 tin
[Page 9] tim thay 50 tin
[Page 10] tim thay 50 tin
[Page 11] tim thay 50 tin
[Page 12] tim thay 50 tin
[Page 13] tim thay 50 tin
[Page 14] tim thay 50 tin
[Page 15] tim thay 50 tin
[Page 16] tim thay 50 tin
[Page 17] tim thay 50 tin
[Page 18] tim thay 50 tin
[Page 19] tim thay 50 tin
[Page 20] tim thay 50 tin
[Page 21] tim thay 50 tin
[Page 22] tim thay 50 tin
[Page 23] tim thay 50 tin
[Page 24] tim thay 50 tin
[Page 25] tim thay 50 tin
[Page 26] tim thay 50 tin
[Page 27] tim thay 50 tin
[Page 28] tim thay 50 tin
[Page 29] tim thay 50 tin
[Page 30] tim thay 50 tin
[Page 31] tim thay 50 tin
[Page 32] tim thay 50 tin
[Page 33] tim thay 50 tin
[Page 34] tim thay 50 tin
[Page 35] tim thay 50 tin
[Page 36] tim thay 50 tin
[Page 37] tim thay 50 tin
[Page 38] tim thay 50 tin
[Page 39] tim thay 50

## 5) Enrich truong sau tu endpoint chi tiet (phap ly, phong, noi that...)

In [ ]:
# Tao dataframe listing ban dau
ndf = pd.DataFrame(rows)
print("Listing shape:", ndf.shape)

# Lay chi tiet tung tin theo list_id de bo sung truong sau
DETAIL_API_TEMPLATE = "https://gateway.chotot.com/v1/public/ad-listing/{list_id}"

# Gioi han so tin enrich de test nhanh. Dat = len(ndf) de enrich full 600
ENRICH_LIMIT = len(ndf)

detail_rows = []
for i, list_id in enumerate(ndf["list_id"].dropna().astype(int).head(ENRICH_LIMIT), start=1):
    try:
        detail_resp = session.get(
            DETAIL_API_TEMPLATE.format(list_id=list_id),
            headers=HEADERS,
            timeout=25,
        )
        if detail_resp.status_code != 200:
            continue

        detail_payload = detail_resp.json()
        ad = detail_payload.get("ad", {})
        params = detail_payload.get("parameters", [])
        param_map = {p.get("id"): p.get("value") for p in params if isinstance(p, dict)}

        detail_rows.append({
            "list_id": ad.get("list_id") or list_id,
            "Gia": ad.get("price"),
            "GiaText": ad.get("price_string"),
            "dienTich": ad.get("size") or ad.get("area"),
            "chieuNgang": ad.get("width"),
            "chieuDai": ad.get("length"),
            "Phongngu": ad.get("rooms"),
            "PhongTam": ad.get("toilets"),
            "SoTang": ad.get("floors"),
            "Loai": param_map.get("house_type"),
            "GiayTo": param_map.get("property_legal_document"),
            "TinhTrangNoiThat": param_map.get("furnishing_sell"),
            "Phuong": param_map.get("ward"),
            "Quan": param_map.get("area"),
            "DiaChi": param_map.get("address"),
            "moTa": ad.get("body"),
            "latitude": ad.get("latitude"),
            "longitude": ad.get("longitude"),
        })

    except Exception:
        continue

    if i % 50 == 0:
        print(f"Da enrich: {i} tin")

    time.sleep(random.uniform(0.25, 0.7))

df_detail = pd.DataFrame(detail_rows).drop_duplicates(subset=["list_id"])
print("Detail shape:", df_detail.shape)

# Gop listing + detail
df_raw = ndf.merge(df_detail, on="list_id", how="left")
print("Merged shape:", df_raw.shape)
df_raw.head()

Listing shape: (2999, 11)
Da enrich: 50 tin
Da enrich: 100 tin
Da enrich: 150 tin
Da enrich: 200 tin
Da enrich: 250 tin
Da enrich: 300 tin
Da enrich: 350 tin
Da enrich: 400 tin
Da enrich: 450 tin
Da enrich: 500 tin
Da enrich: 550 tin
Da enrich: 600 tin
Da enrich: 650 tin
Da enrich: 700 tin
Da enrich: 750 tin
Da enrich: 800 tin
Da enrich: 850 tin
Da enrich: 900 tin
Da enrich: 950 tin
Da enrich: 1000 tin
Da enrich: 1050 tin
Da enrich: 1100 tin
Da enrich: 1150 tin
Da enrich: 1200 tin
Da enrich: 1250 tin
Da enrich: 1300 tin
Da enrich: 1350 tin


In [11]:
print("Shape:", df_raw.shape)
print("\nMissing values:\n", df_raw.isna().sum())

Shape: (600, 28)

Missing values:
 source                0
city                  0
title                 0
price_raw             0
area_raw              0
location_raw          0
detail_url            1
list_id               0
category_name         0
crawl_page            0
crawl_ts              0
Gia                   0
GiaText               0
dienTich              0
chieuNgang          319
chieuDai            416
Phongngu              0
PhongTam            224
SoTang              163
Loai                  0
GiayTo              124
TinhTrangNoiThat    262
Phuong                0
Quan                  0
DiaChi                0
moTa                  0
latitude              0
longitude             0
dtype: int64


## 6) Luu du lieu tho

In [12]:
raw_path = RAW_DIR / "housing_raw_vn_hanoi_deep.csv"
df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")
print(f"Da luu du lieu tho tai: {raw_path}")

Da luu du lieu tho tai: C:\Users\asus\Desktop\DuDoanGiaNha\data\raw\housing_raw_vn_hanoi_deep.csv
